# India's Insurance Industry: Data Analysis Notebook
### Based on: *Unlocking the Potential of India's Insurance Industry: Growth Drivers, Challenges, and Insurtech Pathways*
**Authors:** Eva Aggarwal, Sehdev Singh Saini, Shaurya Mor — Chitkara University, Punjab, India

---
This notebook reconstructs and visualizes the key datasets reported in the research paper, covering:
1. Global benchmarking (penetration & density)
2. Premium income growth (Life vs Non-Life)
3. Claim settlement ratios
4. COVID-19 impact on health insurance
5. InsurTech ecosystem growth
6. Summary scorecard

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#dddddd',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

PALETTE = {
    'india':    '#FF6B35',
    'global':   '#4A90D9',
    'uk':       '#2ECC71',
    'life':     '#3498DB',
    'nonlife':  '#E74C3C',
    'health':   '#9B59B6',
    'insurtech':'#F39C12',
    'neutral':  '#95A5A6',
}

print('✅ Libraries loaded successfully.')

---
## 1. Global Benchmarking: Penetration & Density
> **Source:** IRDAI (2024); Swiss Re Institute (2024); IBEF (2025) — Table 3 of the paper

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────
benchmark = pd.DataFrame({
    'Region':       ['India\n(FY2024)', 'Global\nAverage', 'UK / Developed\nMarkets'],
    'Penetration':  [3.7, 7.0, 11.8],   # % of GDP
    'Density':      [92,  900, 4000],    # USD per capita per year
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('India Insurance vs. Global Benchmarks (FY2024)',
             fontsize=15, fontweight='bold', y=1.02)

colors = [PALETTE['india'], PALETTE['global'], PALETTE['uk']]

# Left: Penetration
ax1 = axes[0]
bars1 = ax1.bar(benchmark['Region'], benchmark['Penetration'],
                color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars1, benchmark['Penetration']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
             f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax1.set_title('Insurance Penetration (% of GDP)', fontweight='bold')
ax1.set_ylabel('% of GDP')
ax1.set_ylim(0, 15)
ax1.axhline(7.0, color=PALETTE['global'], linestyle=':', alpha=0.6, label='Global avg')

# Right: Density (log scale for readability)
ax2 = axes[1]
bars2 = ax2.bar(benchmark['Region'], benchmark['Density'],
                color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars2, benchmark['Density']):
    label = f'${val:,}'
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             label, ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_title('Insurance Density (USD per capita/year)', fontweight='bold')
ax2.set_ylabel('USD per capita')
ax2.set_yscale('log')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))
ax2.set_ylim(10, 8000)

patch_india  = mpatches.Patch(color=PALETTE['india'],  label='India')
patch_global = mpatches.Patch(color=PALETTE['global'], label='Global Average')
patch_uk     = mpatches.Patch(color=PALETTE['uk'],     label='UK / Developed Markets')
fig.legend(handles=[patch_india, patch_global, patch_uk],
           loc='upper center', ncol=3, bbox_to_anchor=(0.5, 0.0), fontsize=10)

plt.tight_layout()
plt.savefig('fig1_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey finding: India\'s penetration is only 53% of the global average,'
      ' and its density is just ~10% of the global average.')

---
## 2. Premium Income Growth — Life vs. Non-Life (FY2018–FY2023)
> **Source:** IRDAI Handbook on Indian Insurance Statistics 2024–25 — Figure 1 reconstruction

In [ ]:
# ── Reconstruct trend data from paper's CAGR figures ──────────────────────
# Life CAGR ≈ 11%, Non-Life CAGR ≈ 12% (implied from paper)
# Base year FY2018: Life ≈ ₹4.58 lakh cr, Non-Life ≈ ₹1.51 lakh cr (IRDAI data)
years     = ['FY2018', 'FY2019', 'FY2020', 'FY2021', 'FY2022', 'FY2023']
life_base = 4.58    # lakh crore INR
nl_base   = 1.51

life_cagr = 0.11
nl_cagr   = 0.12

life_premiums    = [round(life_base * (1 + life_cagr)**i, 2) for i in range(6)]
nonlife_premiums = [round(nl_base   * (1 + nl_cagr  )**i, 2) for i in range(6)]
total_premiums   = [round(l + n, 2) for l, n in zip(life_premiums, nonlife_premiums)]

df_premium = pd.DataFrame({
    'Year': years,
    'Life Insurance':     life_premiums,
    'Non-Life Insurance': nonlife_premiums,
    'Total':              total_premiums,
})

print(df_premium.to_string(index=False))

x     = np.arange(len(years))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
bars_life = ax.bar(x - width/2, life_premiums, width,
                   label='Life Insurance (CAGR ~11%)', color=PALETTE['life'],
                   edgecolor='white')
bars_nl   = ax.bar(x + width/2, nonlife_premiums, width,
                   label='Non-Life Insurance (CAGR ~12%)', color=PALETTE['nonlife'],
                   edgecolor='white')
ax.plot(x, total_premiums, color='#2C3E50', marker='D', linewidth=2,
        markersize=6, label='Total Premium', zorder=5)

for bar in bars_life:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', fontsize=8.5, color='#2C3E50')
for bar in bars_nl:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}', ha='center', fontsize=8.5, color='#2C3E50')

ax.set_title('Insurance Premium Income Growth — Life vs. Non-Life (FY2018–FY2023)',
             fontweight='bold')
ax.set_ylabel('Premium Income (₹ Lakh Crore)')
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.legend(loc='upper left')
ax.set_ylim(0, 12)

ax.annotate('COVID-19 Impact', xy=(3, total_premiums[3]),
            xytext=(3.4, total_premiums[3] + 1.5),
            arrowprops=dict(arrowstyle='->', color='#E74C3C'),
            color='#E74C3C', fontsize=9)

plt.tight_layout()
plt.savefig('fig2_premium_growth.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Claim Settlement Ratios — Leading Life Insurers (FY2021–FY2023)
> **Source:** IRDAI Annual Report 2024 — Table 4 of the paper

In [ ]:
csr = pd.DataFrame({
    'Insurer': ['LIC (Public Sector)', 'HDFC Life', 'ICICI Prudential', 'Max Life', 'SBI Life'],
    'FY2021':  [98.62, 98.01, 97.90, 99.22, 93.09],
    'FY2022':  [98.74, 98.43, 98.13, 99.34, 95.02],
    'FY2023':  [98.76, 99.39, 98.49, 99.65, 96.69],
})

fig, ax = plt.subplots(figsize=(12, 5.5))

x     = np.arange(len(csr['Insurer']))
width = 0.25
fy_colors = ['#BDC3C7', '#7F8C8D', '#2C3E50']

for i, (year, col) in enumerate(zip(['FY2021', 'FY2022', 'FY2023'], fy_colors)):
    bars = ax.bar(x + (i - 1) * width, csr[year], width,
                  label=year, color=col, edgecolor='white')
    for bar, val in zip(bars, csr[year]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() - 1.2,
                f'{val:.2f}%', ha='center', va='top',
                fontsize=7.5, color='white', fontweight='bold')

ax.axhline(98.0, linestyle='--', color=PALETTE['india'], alpha=0.7,
           label='98% reference line')
ax.set_title('Life Insurance Claim Settlement Ratios — Leading Insurers (FY2021–FY2023)',
             fontweight='bold')
ax.set_ylabel('Claim Settlement Ratio (%)')
ax.set_xticks(x)
ax.set_xticklabels(csr['Insurer'], rotation=10, ha='right')
ax.set_ylim(90, 101)
ax.legend()

plt.tight_layout()
plt.savefig('fig3_claim_settlement.png', dpi=150, bbox_inches='tight')
plt.show()

# Stats
print('\nDescriptive statistics (FY2023):')
print(csr[['Insurer','FY2023']].set_index('Insurer').describe().round(2))

---
## 4. COVID-19 Impact on Health Insurance Demand (FY2018–FY2023)
> **Source:** IRDAI Annual Report 2024 — Figure 3 reconstruction

In [ ]:
# Health insurance premiums (₹ thousand crore), anchored by 35% YoY in FY2021
# Pre-COVID growth ~14% pa; post-COVID elevated ~18% pa (sustained demand)
years_h   = ['FY2018', 'FY2019', 'FY2020', 'FY2021', 'FY2022', 'FY2023']
health_p  = [0.39, 0.45, 0.52, 0.70, 0.84, 1.00]   # approx. ₹ lakh crore
yoy_pct   = [None, 15.4, 15.6, 35.0, 20.0, 19.0]

fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax2 = ax1.twinx()

ax1.fill_between(years_h, health_p, alpha=0.18, color=PALETTE['health'])
ax1.plot(years_h, health_p, color=PALETTE['health'], marker='o',
         linewidth=2.5, markersize=8, label='Health Premium (₹ lakh crore)')

for x_val, y_val in zip(years_h, health_p):
    ax1.text(x_val, y_val + 0.015, f'₹{y_val:.2f}L cr',
             ha='center', fontsize=9, color=PALETTE['health'], fontweight='bold')

yoy_vals = [v if v is not None else 0 for v in yoy_pct]
bar_colors = ['#BDC3C7' if v < 25 else '#E74C3C' for v in yoy_vals]
ax2.bar(years_h, yoy_vals, alpha=0.35, color=bar_colors, width=0.4, label='YoY Growth (%)')

ax1.set_title('Health Insurance Premium Growth with COVID-19 Impact (FY2018–FY2023)',
              fontweight='bold')
ax1.set_ylabel('Premium Income (₹ Lakh Crore)', color=PALETTE['health'])
ax2.set_ylabel('YoY Growth (%)', color='#E74C3C')
ax1.set_ylim(0, 1.3)
ax2.set_ylim(0, 55)

ax1.annotate('COVID-19\nDemand Shock\n+35% YoY',
             xy=('FY2021', 0.70), xytext=('FY2019', 0.95),
             arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5),
             color='#E74C3C', fontsize=9, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('fig4_health_covid.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. InsurTech Ecosystem Growth (FY2019–FY2024)
> **Source:** BCG & FICCI (2024); IBEF (2025) — Table 5 & Figure 4 reconstruction

In [ ]:
# Revenue indexed to 1.0 in FY2019 → 12x by FY2024
years_it     = ['FY2019', 'FY2020', 'FY2021', 'FY2022', 'FY2023', 'FY2024']
revenue_idx  = [1.0, 1.5, 2.8, 4.5, 7.5, 12.0]
valuation    = [1.0, 2.5, 4.0, 7.0, 10.5, 13.6]   # USD billion

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('InsurTech Ecosystem Growth — India (FY2019–FY2024)',
             fontsize=14, fontweight='bold')

# Revenue index
ax = axes[0]
ax.fill_between(years_it, revenue_idx, alpha=0.15, color=PALETTE['insurtech'])
ax.plot(years_it, revenue_idx, color=PALETTE['insurtech'],
        marker='s', linewidth=2.5, markersize=8)
for x_val, y_val in zip(years_it, revenue_idx):
    ax.text(x_val, y_val + 0.3, f'{y_val:.1f}x',
            ha='center', fontsize=9.5, fontweight='bold', color=PALETTE['insurtech'])
ax.set_title('Revenue Growth Index (Base = FY2019)', fontweight='bold')
ax.set_ylabel('Revenue Index (1.0 = FY2019)')
ax.set_ylim(0, 14)

# Valuation
ax2 = axes[1]
bars = ax2.bar(years_it, valuation, color=PALETTE['insurtech'],
               edgecolor='white', alpha=0.85)
for bar, val in zip(bars, valuation):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'${val}B', ha='center', fontsize=9, fontweight='bold')
ax2.set_title('Combined InsurTech Valuation (USD Billion)', fontweight='bold')
ax2.set_ylabel('Combined Valuation (USD Billion)')
ax2.set_ylim(0, 17)

ax2.annotate('150+ active\nstartups (2024)',
             xy=('FY2024', 13.6), xytext=('FY2022', 15),
             arrowprops=dict(arrowstyle='->', color='#2C3E50'),
             fontsize=9, color='#2C3E50')

plt.tight_layout()
plt.savefig('fig5_insurtech.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
it_summary = pd.DataFrame({
    'Metric':  ['Active Startups', 'Combined Valuation', 'Revenue Growth (5yr)',
                'UPI Digital Txn Share', 'Aadhaar Coverage'],
    'Value':   ['150+', '~USD 13.6 bn', '12x',
                '>75% of retail txns', '1.3 bn+ individuals'],
    'Source':  ['BCG & FICCI (2024)'] * 3 + ['IBEF (2025)'] * 2,
})
print('\nInsurTech Ecosystem Key Metrics (2024):')
print(it_summary.to_string(index=False))

---
## 6. Structural Barriers Analysis

In [ ]:
barriers = pd.DataFrame({
    'Barrier':   [
        'Rural Coverage Gap\n(only 22% covered)',
        'Language Barriers\n(English/Hindi policies)',
        'Trust Deficit\n(claim fear)',
        'Urban-skewed\nDistribution',
        'Low Awareness\n& Financial Literacy',
        'Informal Economy\nExclusion (~90% workforce)',
    ],
    'Severity':  [9, 7, 8, 7, 8, 9],   # estimated 1–10
    'InsurTech_Mitigation': [6, 7, 6, 7, 5, 5],
    'Category':  ['Access', 'Language', 'Behavioral', 'Access',
                  'Behavioral', 'Structural'],
})

cat_colors = {'Access': '#E74C3C', 'Language': '#3498DB',
              'Behavioral': '#9B59B6', 'Structural': '#F39C12'}
colors = [cat_colors[c] for c in barriers['Category']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Structural Barriers to Insurance Uptake in India',
             fontsize=14, fontweight='bold')

# Horizontal bar — severity
ax = axes[0]
ax.barh(barriers['Barrier'], barriers['Severity'], color=colors,
        edgecolor='white', alpha=0.87)
ax.set_title('Barrier Severity Score (1–10)', fontweight='bold')
ax.set_xlim(0, 11)
ax.set_xlabel('Severity')
for i, v in enumerate(barriers['Severity']):
    ax.text(v + 0.1, i, str(v), va='center', fontweight='bold')

# Scatter — severity vs InsurTech mitigation
ax2 = axes[1]
scatter = ax2.scatter(barriers['Severity'], barriers['InsurTech_Mitigation'],
                      c=colors, s=150, edgecolors='white', linewidth=1.5, zorder=5)
for _, row in barriers.iterrows():
    lbl = row['Barrier'].replace('\n', ' ')
    ax2.annotate(lbl, (row['Severity'], row['InsurTech_Mitigation']),
                 fontsize=7.5, xytext=(4, 4), textcoords='offset points')
ax2.set_title('Severity vs. InsurTech Mitigation Capacity', fontweight='bold')
ax2.set_xlabel('Barrier Severity')
ax2.set_ylabel('InsurTech Mitigation Capacity (1–10)')
ax2.set_xlim(5, 11)
ax2.set_ylim(3, 9)

# Legend
patches = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
fig.legend(handles=patches, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), fontsize=9)

plt.tight_layout()
plt.savefig('fig6_barriers.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(14, 8))
fig.suptitle('India Insurance Industry — Key Findings Summary (2024)',
             fontsize=16, fontweight='bold', y=1.01)
gs = GridSpec(2, 3, figure=fig, hspace=0.55, wspace=0.4)

def kpi_card(ax, value, label, sub, color):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.1), 0.9, 0.8,
                 boxstyle='round,pad=0.03', facecolor=color, alpha=0.12,
                 edgecolor=color, linewidth=2))
    ax.text(0.5, 0.72, value, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color)
    ax.text(0.5, 0.45, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='#2C3E50')
    ax.text(0.5, 0.22, sub, ha='center', va='center',
            fontsize=8, color='#7F8C8D', wrap=True)

kpi_card(fig.add_subplot(gs[0, 0]), '3.7%',   'Insurance Penetration',
         'vs 7% global avg | Source: IRDAI 2024', PALETTE['india'])
kpi_card(fig.add_subplot(gs[0, 1]), '$92',     'Insurance Density',
         'per capita/yr | vs $900 globally',       PALETTE['global'])
kpi_card(fig.add_subplot(gs[0, 2]), '₹7.05L Cr','Total Premium FY2024',
         '~USD 82 bn | +5.6% YoY',                 PALETTE['life'])
kpi_card(fig.add_subplot(gs[1, 0]), '+35%',    'Health Premium Growth',
         'FY2021 COVID shock | IRDAI 2024',         PALETTE['health'])
kpi_card(fig.add_subplot(gs[1, 1]), '$13.6B',  'InsurTech Valuation',
         '150+ startups | BCG & FICCI 2024',        PALETTE['insurtech'])
kpi_card(fig.add_subplot(gs[1, 2]), '22%',     'Rural Coverage',
         'vs 70–80% in developed mkts | NSS 2021',  PALETTE['nonlife'])

plt.savefig('fig7_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Key Conclusions

| # | Finding | Evidence |
|---|---------|----------|
| 1 | India is growing faster than global avg, but penetration gap remains structural | Life CAGR 11% vs global 4%; yet penetration only 3.7% |
| 2 | Trust, not affordability, is the primary barrier | 22% rural coverage despite majority rural population |
| 3 | COVID-19 created a durable demand shift in health insurance | +35% FY2021; sustained growth post-pandemic |
| 4 | InsurTech is a genuine force multiplier | 12x revenue growth; 150+ startups; USD 13.6B valuation |
| 5 | Digital infrastructure (UPI + Aadhaar) uniquely positions India | >75% digital txn share; 1.3B Aadhaar enrollments |
| 6 | Rural and informal sector remain critically underserved | ~90% informal workforce; only 22% rural coverage |

---
**Data Sources:** IRDAI Annual Report 2023–24 | IRDAI Handbook 2024–25 | Swiss Re Institute (2024) | BCG & FICCI (2024) | IBEF (2025) | NSS 75th Round (2021) | PwC India (2024)

*All charts are reconstructed from reported statistics in the paper. Trend series use CAGR/growth rates reported by IRDAI.*